# 00 — Setup catalog

**Purpose:** Create the Unity Catalog structure for the Kitsune pipeline in the development environment (`kitsune_dev`), so dev work stays separate from a future prod catalog.

**Creates:**
- `kitsune_dev` catalog
- `raw` schema
  - `landing` volume (source files)
- `bronze`, `silver`, `gold` schemas (medallion layers)

**Removes:**
- `default` schema (auto-created by Databricks, not used)

**When to run:** Once when setting up the project, or when deploying it to a new workspace. Requires permission to create catalogs.

**Idempotent:** Yes. All statements use `IF NOT EXISTS` / `IF EXISTS`, so re-running is safe.
Note: `IF NOT EXISTS` does **not** update existing objects. To change a comment, use `COMMENT ON`.

In [0]:
%sql
-- Create kitsune_dev Catalog.
CREATE CATALOG IF NOT EXISTS kitsune_dev
COMMENT 'Development environment for the Kitsune anomaly detection project'

In [0]:
%sql
USE CATALOG kitsune_dev;

-- Landing zone: source files exactly as downloaded
CREATE SCHEMA IF NOT EXISTS raw
COMMENT 'Landing zone for source files (CSV.gz) exactly as downloaded from UCI. Files only, no tables.';

-- Medallion layers
CREATE SCHEMA IF NOT EXISTS bronze
COMMENT 'Raw source data loaded as Delta tables, unchanged, plus ingestion metadata (source file, load time). Not cleaned or typed.';

CREATE SCHEMA IF NOT EXISTS silver
COMMENT 'Cleaned and validated data: correct types, no nulls or NaN, features joined with labels. Single source of truth for analysis.';

CREATE SCHEMA IF NOT EXISTS gold
COMMENT 'Model-ready datasets: selected features and chronological train/validation/test splits.';

-- Volume for the original files
CREATE VOLUME IF NOT EXISTS raw.landing
COMMENT 'Original Kitsune SYN DoS files (dataset and labels, CSV.gz) from the UCI ML Repository.';

In [0]:
%sql
-- Databricks creates this automatically; we don't use it
DROP SCHEMA IF EXISTS default;

In [0]:
%sql
SHOW SCHEMAS IN kitsune_dev;

databaseName
bronze
gold
information_schema
raw
silver


In [0]:
%sql
SHOW VOLUMES IN kitsune_dev.raw;

database,volume_name
raw,landing


**Next:** `01_land_source_files`, which copies the raw Kitsune files (dataset and labels) into the `raw.landing` volume.